### RaschPy matrix MFRM worked example

This notebook works through a sample Rasch analysis of a simulated data set (200 persons, 8 items with a maximum score of 5, all rated by 8 raters, no missing data), taking you through the relevant commands step by step, with notes before each cell. Relevant outputs will appear below each cell.

Under the matrix parameterisation, each rater has a full, independent severity for every item x threshold combination — the most flexible (and most heavily parameterised) of the five rater representations.

Import the modules and set the working directory (here called `my_working_directory`) to where you want to save your output files.

In [ ]:
import raschpy as rp
import pandas as pd
import os

os.chdir('my_working_directory')

**A note on data validation**

Every model constructor validates the item-response network automatically at instantiation (`validate=True` by default) and warns if it's disconnected — if there's no chain of persons and items linking every item to every other, the resulting item locations aren't on a common scale, even though `calibrate()` will still run without error. This is worth seeing happen once. Here we build a small, deliberately disconnected data set: two groups of persons who each only answer a disjoint set of items, with no item shared between the groups to link them:

In [ ]:
sim_disconnected = rp.MFRM_Sim_Matrix(no_of_items=8, no_of_persons=40, no_of_raters=4, max_score=5, seed=99)
responses_disconnected = sim_disconnected.responses.copy()
persons = responses_disconnected.index.get_level_values(1).unique()
group_a, group_b = persons[:20], persons[20:]
responses_disconnected.loc[(slice(None), group_a), responses_disconnected.columns[4:]] = float('nan')   # group A: only items 1-4
responses_disconnected.loc[(slice(None), group_b), responses_disconnected.columns[:4]] = float('nan')    # group B: only items 5-8

broken_mfrm = rp.MFRM(responses_disconnected)   # raises a UserWarning

`connectivity_status` records the diagnosis, including which items ended up in which isolated sub-group:

In [ ]:
broken_mfrm.connectivity_status

The rest of this notebook uses a single, fully-connected simulated data set, so this warning won't come up again.

Simulate a data set. Passing `seed=42` makes the simulation fully reproducible — rerunning this notebook will always generate the same data set. 12 items, 8 raters, 200 persons, no missing data.

In [ ]:
sim = rp.MFRM_Sim_Matrix(no_of_items=12, no_of_persons=200, no_of_raters=8, max_score=5, missing=0, seed=42)
sim.responses.to_csv('mfrm_matrix_scores.csv')

If you have your own response data saved to a CSV file instead of simulating it, use `loadup_mfrm_single()` to load and validate it. Demonstrated here by reloading the file we just saved:

In [ ]:
data, invalid_responses = rp.loadup_mfrm_single('mfrm_matrix_scores.csv', max_score=5)

Check the data - view the first two lines

In [ ]:
data.head(2)

Check for any invalid responses (not usable for estimation purposes and excluded)

In [ ]:
invalid_responses

Create an MFRM object. Passing the simulation object `sim` directly (rather than the reloaded `data`) attaches the generating parameters under `mfrm.generating`, which lets us check parameter recovery further down. If you're analysing your own data, pass a DataFrame instead (e.g. `rp.MFRM(data)`).

In [ ]:
mfrm = rp.MFRM(sim)

Calibrate under the matrix parameterisation (a full, independent severity per rater x item x threshold combination). The `%%time` "magic function" returns the time taken to run the cell contents (algorithm run time in this case).

In [ ]:
%%time
mfrm.calibrate_matrix()

Check the item location estimates - view the first two items

In [ ]:
mfrm.items.head(2)

Since this is simulated data, we know the true generating item locations (slm.generating.items) and can check how closely the calibration recovered them. The helper below plots generating vs. estimated values, with an identity line (dashed dark red) and a fitted regression line (dashed red) — the closer the points hug the identity line, the better the recovery. Also displays the Pearson correlation, SD ratio, regression coefficient and RMSE. Recovery for the rater severities will be noisier than the other parameterisations (see below), since the matrix model spreads the same amount of data across many more free parameters (a severity for every rater x item x threshold cell):

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

def recovery_plot(generating, estimated, label, filename):
    x, y = np.asarray(generating), np.asarray(estimated)
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.6)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], color='DarkRed', label='Identity')
    m, b = np.polyfit(x, y, 1)
    ax.plot([lo, hi], [m * lo + b, m * hi + b], color='Red', linestyle='--', label='Regression')
    ax.set_xlabel(f'Generating {label}')
    ax.set_ylabel(f'Estimated {label}')
    ax.set_aspect('equal')
    ax.legend()
    plt.savefig(filename)
    plt.show()
    print(f'{label} Pearson correlation:               {round(np.corrcoef(x, y)[0, 1], 3)}')
    print(f'{label} SD ratio (estimated / generating): {round(y.std() / x.std(), 3)}')
    print(f'{label} regression coefficient:            {round(m, 3)}')
    print(f'{label} RMSE:                              {round(np.sqrt(((x - y) ** 2).mean()), 3)}')

recovery_plot(sim.items, mfrm.items, 'item location', 'my_mfrm_matrix_item_recovery.png')

Generate a table of item statistics (and check run time), and save to file

In [ ]:
%%time
mfrm.item_stats_df_matrix(full=True)
mfrm.item_stats_matrix.to_csv('mfrm_matrix_item_stats.csv')

Check the item statistics table

In [ ]:
mfrm.item_stats_matrix

Generate a table of threshold statistics (and check run time), and save to file

In [ ]:
%%time
mfrm.threshold_stats_df_matrix(full=True)
mfrm.threshold_stats_matrix.to_csv('mfrm_matrix_threshold_stats.csv')

Check the threshold statistics table

In [ ]:
mfrm.threshold_stats_matrix

And the same recovery check for thresholds, against `sim.thresholds`:

In [ ]:
recovery_plot(sim.thresholds, mfrm.thresholds, 'threshold', 'my_mfrm_matrix_threshold_recovery.png')

Generate a table of rater statistics (and check run time), and save to file. By default (`marginal=True`) this reports one row per rater, with item and threshold marginal severities. Pass `marginal=False` for the full (item, threshold) cell-level table instead.

In [ ]:
%%time
mfrm.rater_stats_df_matrix()
mfrm.rater_stats_matrix.to_csv('mfrm_matrix_rater_stats.csv')

Check the rater statistics table

In [ ]:
mfrm.rater_stats_matrix

And the same recovery check for the full rater x item x threshold severities, against `sim.facet_effects`. Since these are per rater x item x threshold, we flatten with `.stack()` first — expect this to be the noisiest of the recovery plots, for the reason noted above:

In [ ]:
orig_severities = sim.facet_effects.stack()
est_severities = mfrm.raters_matrix.stack()
recovery_plot(orig_severities, est_severities, 'rater x item x threshold severity', 'my_mfrm_matrix_rater_recovery.png')

Generate a table of person statistics (and check run time), and save to file

In [ ]:
%%time
mfrm.person_stats_df_matrix(full=True)
mfrm.person_stats_matrix.to_csv('mfrm_matrix_person_stats.csv')

Check the person statistics table - view the first ten persons with `.head(10)`

In [ ]:
mfrm.person_stats_matrix.head(10)

And the same recovery check for person locations, against `sim.persons`:

In [ ]:
recovery_plot(sim.persons, mfrm.persons_matrix, 'person location', 'my_mfrm_matrix_person_recovery.png')

Generate a table of test-level statistics (and check run time), and save to file

In [ ]:
%%time
mfrm.test_stats_df_matrix()
mfrm.test_stats_matrix.to_csv('mfrm_matrix_test_stats.csv')

Check the test statistics table

In [ ]:
mfrm.test_stats_matrix

Run an item residual correlation analysis (and check run time), and save relevant output to file

In [ ]:
%%time
mfrm.item_res_corr_analysis_matrix()
mfrm.item_residual_correlations_matrix.to_csv('mfrm_matrix_item_residual_correlations.csv')
mfrm.item_loadings_matrix.to_csv('mfrm_matrix_item_loadings.csv')

View the table of pairwise standard residual correlations (pairwise local item independence check)

In [ ]:
round(mfrm.item_residual_correlations_matrix, 3)

View the item loadings on the first principal component of the pairwise standard residual correlations (dimensionality test)

In [ ]:
round(mfrm.item_loadings_matrix['PC 1'], 3)

Run a rater residual correlation analysis (and check run time), and save relevant output to file

In [ ]:
%%time
mfrm.rater_res_corr_analysis_matrix()
mfrm.rater_residual_correlations_matrix.to_csv('mfrm_matrix_rater_residual_correlations.csv')
mfrm.rater_loadings_matrix.to_csv('mfrm_matrix_rater_loadings.csv')

View the table of pairwise standard residual correlations (pairwise local rater independence check)

In [ ]:
round(mfrm.rater_residual_correlations_matrix, 3)

View the rater loadings on the first principal component of the pairwise standard residual correlations (dimensionality test)

In [ ]:
round(mfrm.rater_loadings_matrix['PC 1'], 3)

Produce an item characteristic curve (item response function) curve for Item 2, with observed category means plotted and the central item location marked

In [ ]:
mfrm.icc_matrix('Item_2', title='ICC for Item 2', obs=True, central_location=True, cat_highlight=3, xmin=-7, xmax=7, filename='my_mfrm_matrix_icc')

Produce a set of category response curves for Item 2, with category 1 highlighted and the thresholds marked

In [ ]:
mfrm.crcs_matrix('Item_2', thresh_lines=True, obs=[3], cat_highlight=1, xmin=-7, xmax=7, filename='my_mfrm_matrix_crcs')

Produce a set of threshold characteristic curves for Item 2, with observed category means plotted for threshold 5, category 1 highlighted and the thresholds marked

In [ ]:
mfrm.threshold_ccs_matrix('Item_2', thresh_lines=True, obs=[5], cat_highlight=1, xmin=-7, xmax=7, filename='my_mfrm_matrix_threshold_ccs')

Produce an item information function curve for Item 2

In [ ]:
mfrm.iic_matrix('Item_2', title='Information for Item 2', point_info_lines=[0], point_info_labels=True,
                xmin=-7, xmax=7, filename='my_mfrm_matrix_iic')

Produce a test characteristic curve (test response function), with person locations corresponding to scores of 150 and 250 (aggregated across all raters) plotted.

In [ ]:
mfrm.tcc_matrix(score_lines=[150, 250], score_labels=True, xmin=-7, xmax=7, filename='my_mfrm_matrix_tcc')

Produce a test information curve

In [ ]:
mfrm.test_info_matrix(point_info_lines=[0], point_info_labels=True, xmin=-7, xmax=7, filename='my_mfrm_matrix_test_info_curve')

Produce a test CSEM (conditional standard error of measurement) curve, with the CSEM corresponding to a person location of -3 plotted

In [ ]:
mfrm.test_csem_matrix(point_csem_lines=[-3], point_csem_labels=True, ymax=1.4, xmin=-7, xmax=7, filename='my_mfrm_matrix_csem_curve')

Produce a histogram of standardised residuals, with a normal distribution curve overlaid

In [ ]:
mfrm.std_residuals_plot_matrix(bin_width=0.6, normal=True, filename='my_mfrm_matrix_std_residuals_plot')

Now run an anchored analysis, anchoring to the mean severity of Raters 1-4 (e.g. a set of 'gold standard' raters).

In [ ]:
%%time
anchors = ['Rater_1', 'Rater_2', 'Rater_3', 'Rater_4']
mfrm.calibrate_matrix_anchor(anchors)

Check the anchored item locations against the unanchored ones.

In [ ]:
item_check = pd.DataFrame({'Unanchored': mfrm.items,
                           'Anchored': mfrm.anchor_items_matrix})
round(item_check, 3)

Check the anchored thresholds against the unanchored thresholds

In [ ]:
threshold_check = pd.DataFrame({'Unanchored': mfrm.thresholds,
                                'Anchored': mfrm.anchor_thresholds_matrix})
round(threshold_check, 3)

Check the anchored marginal item severities against the unanchored ones - two dataframes

In [ ]:
round(mfrm.marginal_raters_items, 3)

In [ ]:
round(mfrm.anchor_marginal_raters_items, 3)

Check the anchored marginal threshold severities against the unanchored ones - two dataframes

In [ ]:
round(mfrm.marginal_raters_thresholds, 3)

In [ ]:
round(mfrm.anchor_marginal_raters_thresholds, 3)